# Baseline MobileNetV3-Large on PlantVillage: 1-epoch check + T4 timing

This notebook is the first end-to-end run of the training pipeline. It:

1. loads a split manifest (`splits_demo1_13c.csv` for Demo 1, `splits_all_38c.csv` for all 38 classes),
2. builds `mobilenet_v3_large` (ImageNet weights) with a new linear head sized to the number of classes,
3. trains with a **class-weighted** `CrossEntropyLoss` for **1 epoch**, then runs a validation pass,
4. exports a dummy `.onnx` and checks it against PyTorch,
5. records the **exact time per epoch** so we can report the training budget.

**Outputs** (folder `/content/outputs`): one `.pt` checkpoint, one `.onnx` and one `timing_*.json` per configuration.

## 1. Setup

Install the ONNX packages (not preinstalled on Colab), import everything in one place and check that a GPU is attached.

In [ ]:
import subprocess
import sys

pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript"],
    capture_output=True,
    text=True,
)
print("pip exit code:", pip.returncode, pip.stderr[-300:])

In [ ]:
import gc
import json
import os
import random
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from IPython.display import display
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "No GPU found: Runtime > Change runtime type > T4 GPU"
device = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME, "| torch", torch.__version__)
if "T4" not in GPU_NAME:
    print("WARNING: timings are only valid for a T4, this runtime has", GPU_NAME)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Configuration

Everything you may want to change is in this cell.

* `CONFIGS` maps a config name to its manifest and to the column used as the class label
  (`demo1_id` is 0..12 for the 13 Demo 1 classes, `class_id` is 0..37 for all classes).
* `CONFIGS_TO_RUN` is the toggle: keep both to get both timings in one run, or use `["demo1_13c"]` for a quick run.
* `REVISION` pins the Hugging Face dataset commit, so the images match the manifests.
* `HP["epochs"] = 1` is deliberate for this task: the point is a working pipeline and an exact epoch time.

In [ ]:
REVISION = "9e97599868962bd0079b8db4b7f1efa9185fa1e7"

CONFIGS = {
    "demo1_13c": {"manifest": "splits_demo1_13c.csv", "target": "demo1_id"},
    "all_38c": {"manifest": "splits_all_38c.csv", "target": "class_id"},
}
CONFIGS_TO_RUN = ["demo1_13c", "all_38c"]

HP = {
    "epochs": 1,
    "batch_size": 64,
    "lr": 5e-4,
    "weight_decay": 1e-4,
    "img_size": 224,
    "workers": min(os.cpu_count() or 2, 8),
    "amp": True,
    "seed": 42,
}

WORK = Path("/content/work")
MANIFEST_DIR = WORK / "datasets"
LABELS_PATH = WORK / "labels.csv"
IMG_ROOT = Path("/content/plantvillage")
OUT_DIR = Path("/content/outputs")
for d in (MANIFEST_DIR, IMG_ROOT, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(HP["seed"])
np.random.seed(HP["seed"])
torch.manual_seed(HP["seed"])
torch.backends.cudnn.benchmark = True

## 3. Load the manifests and `labels.csv`

For uploading the files, the notebook must be run on the browser. Alternative would be to upload the files to drive and then point the path to the one on the drive

In [ ]:
from google.colab import files

needed = [MANIFEST_DIR / CONFIGS[c]["manifest"] for c in CONFIGS_TO_RUN]
missing = [p for p in needed if not p.exists()]

if missing:
    print("Select these files:", [p.name for p in missing])
    for fname, data in files.upload().items():
        target_path = MANIFEST_DIR / fname
        target_path.write_bytes(data)
if not LABELS_PATH.exists():
    print("Select labels.csv file")
    for _fname, data in files.upload().items():
        target_path = LABELS_PATH
        target_path.write_bytes(data)
for p in needed:
    assert p.exists(), f"missing {p}"

labels = pd.read_csv(LABELS_PATH)
manifests = {c: pd.read_csv(MANIFEST_DIR / CONFIGS[c]["manifest"]) for c in CONFIGS_TO_RUN}
for c, m in manifests.items():
    print(c, len(m), "images", m["split"].value_counts().to_dict())
display(labels[["class_id", "demo1_id", "crop", "disease"]])

## 4. Download and extract the images

The manifests only list file paths. The images come from `data.zip` (about 2.2 GB) in the
`mohanty/PlantVillage` dataset repo. We extract **only the files the manifests need**, to local disk.
This step is not part of the epoch timing.

In [ ]:
t0 = time.perf_counter()
zip_path = hf_hub_download(
    "mohanty/PlantVillage", "data.zip", repo_type="dataset", revision=REVISION
)
print(f"data.zip ready in {time.perf_counter() - t0:.0f}s")

wanted = sorted(set().union(*(set(m["path"]) for m in manifests.values())))
todo = [p for p in wanted if not (IMG_ROOT / p).exists()]
print(f"{len(wanted)} images needed, {len(todo)} still to extract")
if todo:
    with zipfile.ZipFile(zip_path) as zf:
        for p in tqdm(todo):
            zf.extract(p, IMG_ROOT)
assert all((IMG_ROOT / p).exists() for p in wanted), "some images are missing after extraction"

## 5. Dataset and data loaders

Images are 256x256 JPEGs. Training uses a random crop/flip, validation a plain resize to 224x224,
both normalised with the ImageNet statistics the pretrained weights expect.
The path column in the manifest is relative to `IMG_ROOT`.

In [10]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
SIZE = HP["img_size"]

train_tfm = transforms.Compose(
    [
        transforms.RandomResizedCrop(SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)
eval_tfm = transforms.Compose(
    [
        transforms.Resize((SIZE, SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)


class LeafDataset(Dataset):
    def __init__(self, df, target_col, tfm):
        self.paths = [str(IMG_ROOT / p) for p in df["path"]]
        self.targets = df[target_col].astype(int).tolist()
        self.tfm = tfm

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as img:
            x = self.tfm(img.convert("RGB"))
        return x, self.targets[i]


def make_loader(df, target_col, train):
    return DataLoader(
        LeafDataset(df, target_col, train_tfm if train else eval_tfm),
        batch_size=HP["batch_size"],
        shuffle=train,
        num_workers=HP["workers"],
        pin_memory=True,
        persistent_workers=HP["workers"] > 0,
        drop_last=train,
    )

## 6. Model and class weights

* `build_model` loads `mobilenet_v3_large(weights="DEFAULT")` and replaces the last `Linear` layer
  (the 1000-class ImageNet head) with a new one of size `num_classes`.
* `class_weights` computes inverse-frequency weights from the **training split only**:
  `weight_c = N / (K * n_c)`. A class with few images (Potato healthy, 152) gets a large weight,
  a class with many (Tomato yellow leaf curl virus, 5,375) a small one, so rare classes are not ignored.
  The weights are passed to `CrossEntropyLoss(weight=...)`.

In [11]:
def build_model(num_classes, pretrained=True):
    model = models.mobilenet_v3_large(weights="DEFAULT" if pretrained else None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model


def class_weights(train_targets, num_classes):
    counts = np.bincount(train_targets, minlength=num_classes)
    assert (counts > 0).all(), "a class has no training images"
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32), counts

## 7. Training and evaluation functions (with timing)

Both functions call `torch.cuda.synchronize()` before and after timing, because CUDA runs asynchronously
and without it the clock would stop before the GPU is done.

* `train_one_epoch` also records `data_wait_sec`: time the training loop spent waiting for the next batch.
  If it is a large share of the epoch, the Colab CPUs (JPEG decoding) are the bottleneck, not the T4.
* `evaluate` reports an unweighted validation loss, accuracy and macro F1 (macro F1 matters here because of the class imbalance).

In [12]:
def sync():
    torch.cuda.synchronize()


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum = torch.zeros((), device=device)
    n, data_wait = 0, 0.0
    sync()
    t_start = t_fetch = time.perf_counter()
    for x, y in loader:
        data_wait += time.perf_counter() - t_fetch
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=HP["amp"]):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.detach() * len(y)
        n += len(y)
        t_fetch = time.perf_counter()
    sync()
    return {
        "loss": float(loss_sum) / n,
        "seconds": time.perf_counter() - t_start,
        "data_wait_sec": data_wait,
        "images": n,
    }


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum = torch.zeros((), device=device)
    preds, targets = [], []
    sync()
    t_start = time.perf_counter()
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=HP["amp"]):
            logits = model(x)
        loss_sum += F.cross_entropy(logits.float(), y, reduction="sum")
        preds.append(logits.argmax(1).cpu())
        targets.append(y.cpu())
    sync()
    seconds = time.perf_counter() - t_start
    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return {
        "loss": float(loss_sum) / len(targets),
        "acc": float((preds == targets).mean()),
        "macro_f1": float(f1_score(targets, preds, average="macro")),
        "seconds": seconds,
        "images": len(targets),
    }

## 8. Run: 1 epoch per configuration
 trains 1 epoch and runs the validation pass, timing the whole epoch (train + validation),
3. saves the checkpoint `mobilenetv3_<config>_epoch1.pt` and the metrics `timing_<config>.json`.

!! The first epoch includes one-off costs so it is slightly
slower than later epochs.

In [ ]:
def run_config(name):
    target = CONFIGS[name]["target"]
    df = manifests[name]
    train_df, val_df = df[df["split"] == "train"], df[df["split"] == "val"]
    ids = sorted(df[target].astype(int).unique())

    num_classes = len(ids)

    assert ids == list(range(num_classes)), f"Expected class IDs 0..{num_classes - 1}, got {ids}"

    first = df.drop_duplicates(target).sort_values(target)

    class_names = (first["crop"] + "___" + first["disease"]).tolist()

    assert len(class_names) == num_classes

    weights, counts = class_weights(train_df[target].astype(int).to_numpy(), num_classes)
    table = pd.DataFrame(
        {"class": class_names, "train_images": counts, "weight": weights.numpy().round(2)}
    )
    print(f"=== {name}: {num_classes} classes, {len(train_df)} train / {len(val_df)} val ===")
    display(table)

    model = build_model(num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=HP["amp"])
    train_loader = make_loader(train_df, target, train=True)
    val_loader = make_loader(val_df, target, train=False)

    torch.cuda.reset_peak_memory_stats()
    epochs = []
    for epoch in range(1, HP["epochs"] + 1):
        sync()
        t0 = time.perf_counter()
        tr = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        va = evaluate(model, val_loader)
        sync()
        epoch_sec = time.perf_counter() - t0
        epochs.append({"epoch": epoch, "epoch_sec": epoch_sec, "train": tr, "val": va})
        print(
            f"epoch {epoch}: {epoch_sec:.1f}s total "
            f"(train {tr['seconds']:.1f}s + val {va['seconds']:.1f}s) | "
            f"train loss {tr['loss']:.3f} | val acc {va['acc']:.3f} | "
            f"val macro F1 {va['macro_f1']:.3f}"
        )

    ckpt_path = OUT_DIR / f"mobilenetv3_{name}_epoch{HP['epochs']}.pt"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "num_classes": num_classes,
            "class_names": class_names,
            "config": name,
            "epochs_trained": HP["epochs"],
            "img_size": SIZE,
            "mean": MEAN,
            "std": STD,
        },
        ckpt_path,
    )

    last = epochs[-1]
    record = {
        "config": name,
        "num_classes": num_classes,
        "train_images": len(train_df),
        "val_images": len(val_df),
        "sec_per_epoch": float(np.mean([e["epoch_sec"] for e in epochs])),
        "train_sec": last["train"]["seconds"],
        "val_sec": last["val"]["seconds"],
        "train_img_per_s": last["train"]["images"] / last["train"]["seconds"],
        "data_wait_pct": 100 * last["train"]["data_wait_sec"] / last["train"]["seconds"],
        "peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9,
        "val_acc": last["val"]["acc"],
        "val_macro_f1": last["val"]["macro_f1"],
        "gpu": GPU_NAME,
        "torch": torch.__version__,
        "hp": HP,
        "epochs": epochs,
    }
    (OUT_DIR / f"timing_{name}.json").write_text(json.dumps(record, indent=2))

    del train_loader, val_loader, model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return record


RESULTS = {name: run_config(name) for name in CONFIGS_TO_RUN}

## 9. Dummy ONNX export check

For every checkpoint from section 8 this cell rebuilds the model on CPU, exports it to ONNX with a dynamic batch axis,
validates the file with `onnx.checker`, runs it with `onnxruntime` on a batch of 2 and asserts that the output
matches PyTorch (max difference below 1e-3). This proves the exported head and the 224x224 input shape work end to end.

In [ ]:
def onnx_check(name):
    import onnx
    import onnxruntime as ort

    ckpt = torch.load(OUT_DIR / f"mobilenetv3_{name}_epoch{HP['epochs']}.pt", map_location="cpu")
    model = build_model(ckpt["num_classes"], pretrained=False)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    onnx_path = OUT_DIR / f"mobilenetv3_{name}.onnx"
    dummy = torch.randn(1, 3, SIZE, SIZE)
    export_kwargs = {
        "input_names": ["input"],
        "output_names": ["logits"],
        "dynamic_axes": {"input": {0: "batch"}, "logits": {0: "batch"}},
        "opset_version": 17,
    }
    try:
        torch.onnx.export(model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
    except TypeError:  # older torch has no `dynamo` argument
        torch.onnx.export(model, dummy, str(onnx_path), **export_kwargs)

    onnx.checker.check_model(onnx.load(str(onnx_path)))
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    batch = torch.randn(2, 3, SIZE, SIZE)  # batch of 2 also checks the dynamic batch axis
    with torch.no_grad():
        expected = model(batch).numpy()
    got = session.run(None, {"input": batch.numpy()})[0]
    max_diff = float(np.abs(expected - got).max())
    assert got.shape == (2, ckpt["num_classes"]), got.shape
    assert max_diff < 1e-3, f"ONNX output differs from PyTorch by {max_diff}"
    print(f"{name}: ONNX ok | {onnx_path.stat().st_size / 1e6:.1f} MB | max diff {max_diff:.2e}")


for name in CONFIGS_TO_RUN:
    onnx_check(name)

## 10. Timing summary and training budget

Reads every `timing_*.json` in the output folder and prints one row per configuration.
The last three columns project the training time for 10, 20 and 30 epochs from the measured epoch time
(a simple multiplication, so it is only as good as the one-epoch measurement).

How to read it:

* `sec_per_epoch` is train + validation time on the T4: **the number to report to the mentor**.
* `data_wait_pct` close to 0 means the GPU is the bottleneck; a high value means the loader (Colab CPUs) is.
* The `val_acc` and `val_macro_f1` after one epoch only show that the pipeline learns, they are not the final model quality.

In [ ]:
rows = []
for path in sorted(OUT_DIR.glob("timing_*.json")):
    r = json.loads(path.read_text())
    rows.append(
        {
            "config": r["config"],
            "classes": r["num_classes"],
            "train_imgs": r["train_images"],
            "val_imgs": r["val_images"],
            "sec_per_epoch": round(r["sec_per_epoch"], 1),
            "train_sec": round(r["train_sec"], 1),
            "val_sec": round(r["val_sec"], 1),
            "train_img_per_s": round(r["train_img_per_s"]),
            "data_wait_pct": round(r["data_wait_pct"]),
            "peak_mem_gb": round(r["peak_mem_gb"], 2),
            "val_acc": round(r["val_acc"], 3),
            "val_macro_f1": round(r["val_macro_f1"], 3),
            "gpu": r["gpu"],
        }
    )
summary = pd.DataFrame(rows)
for n_epochs in (10, 20, 30):
    summary[f"{n_epochs} epochs (min)"] = (summary["sec_per_epoch"] * n_epochs / 60).round(1)
display(summary)

## 11. Keep the results

Everything is in `/content/outputs`, which disappears when the Colab runtime is recycled.
Set `USE_DRIVE = True` to copy it to Google Drive, otherwise the folder is zipped for download.

In [ ]:
USE_DRIVE = False

print(sorted(p.name for p in OUT_DIR.iterdir()))
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/rootine/baseline_mobilenetv3")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
    print("copied to", dest)
else:
    from google.colab import files

    archive = shutil.make_archive("/content/baseline_outputs", "zip", OUT_DIR)
    files.download(archive)